# Oracle SGA Architecture: Buffer Cache & Library Cache Parsing

Interactive hands-on sandbox exploring internal storage mechanics, algorithmic data structures, and architectural invariants.


In [ ]:
import sys
from pathlib import Path

# Prepend project_solution to sys.path so Track A internal engines import cleanly
ps_dir = Path('.').resolve() / 'project_solution'
if str(ps_dir) not in sys.path:
    sys.path.insert(0, str(ps_dir))

print(f'Python runtime: {sys.version.split()[0]}')
print('Loaded internal mechanics for: Module_07_Oracle_Database_Architecture_SGA_PGA')


## 1. Engine Initialization & Setup

Importing the module's Track A internal simulation engine and instantiating state.


In [ ]:
from oracle_sga_engine import DatabaseBufferCache, LibraryCache

# Initialize Oracle SGA Subsystems
lib_cache = LibraryCache()
buf_cache = DatabaseBufferCache(capacity_blocks=50)

print(f"SGA Shared Pool initialized. Buffer cache capacity: {buf_cache.capacity} blocks.")


## 2. Core Architectural Operations & State Mutation

Executing data mutations, transactions, or indexing procedures.


In [ ]:
# Hard Parse vs Soft Parse in Oracle Library Cache:
# Hard Parse: Generates syntax tree, semantic check, and Cost-Based Optimizer execution plan.
# Soft Parse: Reuses existing plan hash when SQL text matches identically (Bind Variables).
bind_sql = "SELECT * FROM emp WHERE emp_id = :1"

plan_1, was_soft_1 = lib_cache.parse_and_get_plan("SELECT * FROM emp WHERE emp_id = 101", bind_normalized_sql=bind_sql)
plan_2, was_soft_2 = lib_cache.parse_and_get_plan("SELECT * FROM emp WHERE emp_id = 999", bind_normalized_sql=bind_sql)

print("First execution (Soft Parse?):", was_soft_1, f"Total Hard Parses: {lib_cache.hard_parses}")
print("Second execution with bind variable (Soft Parse?):", was_soft_2, f"Total Soft Parses: {lib_cache.soft_parses}")


## 3. Performance Micro-Benchmarking & Invariant Verification

Evaluating execution latency, cache hits, or computational trade-offs.


In [ ]:
# Buffer Cache Touch Count & Aging (Touch-Count algorithm)
buf_cache.access_block(100)
buf_cache.access_block(100) # Increments touch count
buf_cache.access_block(101)

print("Buffer cache active block count:", len(buf_cache._cache))
print("Block 100 touch count:", buf_cache._cache[100].touch_count)


## 4. Architectural Invariant Verification

Asserting mathematical correctness and durability invariants.


In [ ]:
# Verify Oracle SGA Invariants
assert lib_cache.hard_parses == 1, "Bind variables must eliminate duplicate hard parses"
assert lib_cache.soft_parses == 1, "Cursor sharing must achieve soft parse"
assert buf_cache._cache[100].touch_count >= 2
assert buf_cache.capacity == 50
print("[+] Oracle SGA Library Cache and Buffer Cache invariants verified successfully!")


## Summary & Operational DBRE Best Practices

1. **Never bypass serialization contracts:** Always enforce binary-safe schemas and validated boundaries.
2. **Monitor buffer and memory allocations:** Understand the latency cliff when in-memory structures spill to disk.
3. **Ensure idempotency across replication tiers:** Distributed operations must survive retries without corrupting state.
